<a href="https://colab.research.google.com/github/M-Arya-Rejoy-04/Loan-Eligibility-PySpark/blob/main/Loan_Eligibility_Prediction_Using_PySpark.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# -*- coding: utf-8 -*-
"""Loan Eligibility.py

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/1R4pP_naatS5A2n_WrkhIrC6_oTrXIwgo
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.ml.feature import StringIndexer, VectorAssembler, Imputer
from pyspark.ml.classification import RandomForestClassifier, GBTClassifier, LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
from pyspark.ml import Pipeline
import warnings
warnings.filterwarnings('ignore')

# Initialize Spark Session
spark = SparkSession.builder \
    .appName("LoanEligibilityPrediction") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true") \
    .getOrCreate()

# Load the datasets
def load_data():
    # Load training data
    train_df = pd.read_csv("/content/drive/MyDrive/CSVs/loan-train.csv")

    # Load test data (without Loan_Status)
    test_df = pd.read_csv("/content/drive/MyDrive/CSVs/loan-test.csv")

    print("Training data shape:", train_df.shape)
    print("Test data shape:", test_df.shape)

    return train_df, test_df

train_pd, test_pd = load_data()

# Convert to Spark DataFrames
train_spark = spark.createDataFrame(train_pd)
test_spark = spark.createDataFrame(test_pd)

print("=== DATA OVERVIEW ===")
print("Training Data Schema:")
train_spark.printSchema()
print("\nTraining Data Sample:")
train_spark.show(5)
print("\nTest Data Sample:")
test_spark.show(5)

# Data Exploration and Analysis
print("=== DATA EXPLORATION ===")

# Check for missing values
def check_missing_data(df, df_name):
    print(f"\nMissing values in {df_name}:")
    for col in df.columns:
        null_count = df.filter(df[col].isNull()).count()
        if null_count > 0:
            print(f"{col}: {null_count} ({null_count/df.count()*100:.2f}%)")

check_missing_data(train_spark, "Training Data")
check_missing_data(test_spark, "Test Data")

# Target variable distribution
print("\nTarget Variable Distribution:")
train_spark.groupBy("Loan_Status").count().show()

# Basic statistics for numerical columns
print("\nNumerical Features Statistics:")
numerical_cols = ['ApplicantIncome', 'CoapplicantIncome', 'LoanAmount', 'Loan_Amount_Term', 'Credit_History']
train_spark.select(numerical_cols).describe().show()

# Data Preprocessing and Feature Engineering
print("=== DATA PREPROCESSING ===")

class LoanDataPreprocessor:
    def __init__(self):
        self.categorical_cols = ['Gender', 'Married', 'Dependents', 'Education',
                               'Self_Employed', 'Property_Area']
        self.numerical_cols = ['ApplicantIncome', 'CoapplicantIncome', 'LoanAmount',
                             'Loan_Amount_Term', 'Credit_History']

    def preprocess(self, df, is_training=True):
        # Create a copy to avoid modifying original
        processed_df = df

        # Handle 'Dependents' column - replace '3+' with '3'
        processed_df = processed_df.withColumn(
            'Dependents',
            when(col('Dependents') == '3+', '3').otherwise(col('Dependents')))

        # Create new features
        processed_df = self.create_new_features(processed_df)

        # Handle missing values
        processed_df = self.handle_missing_values(processed_df)

        return processed_df

    def create_new_features(self, df):
        # Total Income
        df = df.withColumn('TotalIncome',
                         col('ApplicantIncome') + col('CoapplicantIncome'))

        # Income to Loan Ratio
        df = df.withColumn('IncomeToLoanRatio',
                         when(col('LoanAmount') > 0,
                              col('TotalIncome') / col('LoanAmount'))
                         .otherwise(0))

        # EMI (assuming 9% interest rate)
        df = df.withColumn('EMI',
                         when((col('LoanAmount') > 0) & (col('Loan_Amount_Term') > 0),
                              (col('LoanAmount') * 0.09 * pow(1.09, col('Loan_Amount_Term')/12)) /
                              (pow(1.09, col('Loan_Amount_Term')/12) - 1))
                         .otherwise(0))

        # EMI to Income Ratio
        df = df.withColumn('EMItoIncomeRatio',
                         when(col('TotalIncome') > 0,
                              col('EMI') / col('TotalIncome'))
                         .otherwise(0))

        # Has Coapplicant
        df = df.withColumn('HasCoapplicant',
                         when(col('CoapplicantIncome') > 0, 1).otherwise(0))

        return df

    def handle_missing_values(self, df):
        # For categorical columns, fill with mode
        categorical_columns = ['Gender', 'Married', 'Dependents', 'Self_Employed']

        for col_name in categorical_columns:
            # Get mode (most frequent value)
            mode_row = df.groupBy(col_name).count().orderBy(desc('count')).first()
            if mode_row:
                mode_value = mode_row[col_name]
                df = df.fillna({col_name: mode_value})

        # For numerical columns, we'll use imputer in the pipeline
        return df
# Initialize preprocessor and preprocess data
preprocessor = LoanDataPreprocessor()
train_processed = preprocessor.preprocess(train_spark, is_training=True)
test_processed = preprocessor.preprocess(test_spark, is_training=False)

print("Processed Training Data Sample:")
train_processed.select('Loan_ID', 'TotalIncome', 'IncomeToLoanRatio', 'EMI', 'EMItoIncomeRatio', 'HasCoapplicant').show(5)

# Prepare features for ML pipeline
print("=== FEATURE ENGINEERING ===")

# Define categorical and numerical columns
categorical_cols = ['Gender', 'Married', 'Dependents', 'Education', 'Self_Employed', 'Property_Area']
numerical_cols = ['ApplicantIncome', 'CoapplicantIncome', 'LoanAmount', 'Loan_Amount_Term',
                 'Credit_History', 'TotalIncome', 'IncomeToLoanRatio', 'EMI', 'EMItoIncomeRatio', 'HasCoapplicant']

# Create StringIndexers for categorical columns
indexers = [StringIndexer(inputCol=col, outputCol=col+"_index", handleInvalid="keep")
           for col in categorical_cols]

# Imputer for numerical columns
imputer = Imputer(inputCols=numerical_cols,
                outputCols=[col+"_imputed" for col in numerical_cols])

# Assemble all features
feature_cols = [col+"_index" for col in categorical_cols] + [col+"_imputed" for col in numerical_cols]
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")

# StringIndexer for target variable
label_indexer = StringIndexer(inputCol="Loan_Status", outputCol="label")
# Build Machine Learning Models
print("=== MODEL TRAINING ===")

# Define models
rf = RandomForestClassifier(featuresCol="features", labelCol="label", seed=42)
gbt = GBTClassifier(featuresCol="features", labelCol="label", seed=42)
lr = LogisticRegression(featuresCol="features", labelCol="label")

# Create pipelines
rf_pipeline = Pipeline(stages=indexers + [imputer, assembler, label_indexer, rf])
gbt_pipeline = Pipeline(stages=indexers + [imputer, assembler, label_indexer, gbt])
lr_pipeline = Pipeline(stages=indexers + [imputer, assembler, label_indexer, lr])
# Split data for training and validation
train_data, validation_data = train_processed.randomSplit([0.8, 0.2], seed=42)

print(f"Training data count: {train_data.count()}")
print(f"Validation data count: {validation_data.count()}")

# Train models
print("Training Random Forest...")
rf_model = rf_pipeline.fit(train_data)

print("Training Gradient Boosting...")
gbt_model = gbt_pipeline.fit(train_data)

print("Training Logistic Regression...")
lr_model = lr_pipeline.fit(train_data)

# Model Evaluation
print("=== MODEL EVALUATION ===")

def evaluate_model(model, data, model_name):
    predictions = model.transform(data)

    # Calculate metrics
    evaluator_auc = BinaryClassificationEvaluator(labelCol="label")
    evaluator_accuracy = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")
    evaluator_precision = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="weightedPrecision")
    evaluator_recall = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="weightedRecall")
    evaluator_f1 = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="f1")

    auc = evaluator_auc.evaluate(predictions)
    accuracy = evaluator_accuracy.evaluate(predictions)
    precision = evaluator_precision.evaluate(predictions)
    recall = evaluator_recall.evaluate(predictions)
    f1 = evaluator_f1.evaluate(predictions)

    print(f"\n{model_name} Performance:")
    print(f"AUC: {auc:.4f}")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1-Score: {f1:.4f}")

    # Confusion Matrix
    confusion_matrix = predictions.groupBy("label", "prediction").count().orderBy("label", "prediction")
    print(f"\nConfusion Matrix for {model_name}:")
    confusion_matrix.show()

    return predictions, {
        'AUC': auc,
        'Accuracy': accuracy,
        'Precision': precision,
        'Recall': recall,
        'F1-Score': f1
    }

# Evaluate all models
rf_predictions, rf_metrics = evaluate_model(rf_model, validation_data, "Random Forest")
gbt_predictions, gbt_metrics = evaluate_model(gbt_model, validation_data, "Gradient Boosting")
lr_predictions, lr_metrics = evaluate_model(lr_model, validation_data, "Logistic Regression")

# Compare models
metrics_df = pd.DataFrame({
    'Random Forest': rf_metrics,
    'Gradient Boosting': gbt_metrics,
    'Logistic Regression': lr_metrics
})

print("\n=== MODEL COMPARISON ===")
print(metrics_df)

# Feature Importance for Random Forest
print("\n=== FEATURE IMPORTANCE (Random Forest) ===")
rf_feature_importance = rf_model.stages[-1].featureImportances
feature_importance_df = pd.DataFrame({
    'feature': feature_cols,
    'importance': rf_feature_importance.toArray()
}).sort_values('importance', ascending=False)

print("Top 10 Most Important Features:")
print(feature_importance_df.head(10))

# Make predictions on test data
print("=== PREDICTIONS ON TEST DATA ===")

# Use the best model (Random Forest based on typical performance)
best_model = rf_model
test_predictions = best_model.transform(test_processed)

# Convert predictions back to original labels
index_to_label = {0.0: 'Y', 1.0: 'N'}  # Based on how StringIndexer encodes
test_results = test_predictions.select(
    'Loan_ID',
    'prediction'
).withColumn(
    'Loan_Status_Predicted',
    when(col('prediction') == 0.0, 'Y').otherwise('N')
)

print("Test Predictions Sample:")
test_results.show(10)

# Save results
test_results_pd = test_results.toPandas()
test_results_pd[['Loan_ID', 'Loan_Status_Predicted']].to_csv('loan_predictions.csv', index=False)
print("\nPredictions saved to 'loan_predictions.csv'")

# Business Insights and Analysis
print("=== BUSINESS INSIGHTS ===")

# 1. Approval Rate
total_applications = test_results.count()
approved = test_results.filter(col('Loan_Status_Predicted') == 'Y').count()
approval_rate = (approved / total_applications) * 100

print(f"\n1. Overall Approval Rate: {approval_rate:.2f}%")
print(f"   - Approved: {approved}")
print(f"   - Rejected: {total_applications - approved}")

# 2. Key Factors Analysis
feature_impact = feature_importance_df.head(10)
print("\n2. Top Factors Influencing Loan Approval:")
for idx, row in feature_impact.iterrows():
    print(f"   - {row['feature']}: {row['importance']:.4f}")

# 3. Income Analysis
income_stats = test_processed.select(
    mean('ApplicantIncome').alias('avg_income'),
    mean('TotalIncome').alias('avg_total_income'),
    mean('LoanAmount').alias('avg_loan_amount')
).collect()[0]

print(f"\n3. Income and Loan Statistics:")
print(f"   - Average Applicant Income: ${income_stats['avg_income']:.2f}")
print(f"   - Average Total Income: ${income_stats['avg_total_income']:.2f}")
print(f"   - Average Loan Amount: ${income_stats['avg_loan_amount']:.2f}")

# 4. Credit History Impact
if 'Credit_History' in [col.name for col in test_processed.schema]:
    credit_analysis = test_processed.filter(col('Credit_History').isNotNull()).groupBy('Credit_History').count()
    print("\n4. Credit History Distribution:")
    credit_analysis.show()

# Visualization of Results (Optional - would require matplotlib)
def create_visualizations():
    # This function would create various plots and charts
    # For a complete implementation, you would add matplotlib/seaborn visualizations
    pass

# Model Deployment Preparation
print("=== MODEL DEPLOYMENT PREPARATION ===")

# Save the trained model
best_model.write().overwrite().save("loan_eligibility_model")
print("Model saved to 'loan_eligibility_model' directory")

# Create a prediction function for new data
def predict_loan_eligibility(new_application):
    """
    Function to predict loan eligibility for new applications
    """
    # Convert to Spark DataFrame
    new_df = spark.createDataFrame([new_application])

    # Preprocess
    processed_new = preprocessor.preprocess(new_df, is_training=False)

    # Make prediction
    prediction = best_model.transform(processed_new)

    # Get result
    result = prediction.select(
        'Loan_ID',
        when(col('prediction') == 0.0, 'Approved').otherwise('Rejected').alias('Status'),
        col('probability')
    ).collect()[0]

    return result

# Example prediction
example_application = {
    'Loan_ID': 'LP002999',
    'Gender': 'Male',
    'Married': 'Yes',
    'Dependents': '0',
    'Education': 'Graduate',
    'Self_Employed': 'No',
    'ApplicantIncome': 5000,
    'CoapplicantIncome': 2000,
    'LoanAmount': 150,
    'Loan_Amount_Term': 360,
    'Credit_History': 1,
    'Property_Area': 'Urban'
}

print("\n=== EXAMPLE PREDICTION ===")
try:
    example_result = predict_loan_eligibility(example_application)
    print(f"Loan ID: {example_result['Loan_ID']}")
    print(f"Status: {example_result['Status']}")
    print(f"Probability: {example_result['probability']}")
except Exception as e:
    print(f"Example prediction failed: {e}")

# Performance Summary
print("\n=== PROJECT SUMMARY ===")
print("✓ Data loaded and preprocessed successfully")
print("✓ Feature engineering completed")
print("✓ Multiple ML models trained and evaluated")
print("✓ Best model selected (Random Forest)")
print("✓ Predictions generated for test data")
print("✓ Business insights extracted")
print("✓ Model saved for deployment")

print(f"\nFinal Model Performance on Validation Data:")
print(f"Accuracy: {rf_metrics['Accuracy']:.4f}")
print(f"AUC: {rf_metrics['AUC']:.4f}")

# Stop Spark session
spark.stop()
print("\nSpark session stopped. Project completed successfully!")


Training data shape: (614, 13)
Test data shape: (367, 12)
=== DATA OVERVIEW ===
Training Data Schema:
root
 |-- Loan_ID: string (nullable = true)
 |-- Gender: string (nullable = true)
 |-- Married: string (nullable = true)
 |-- Dependents: string (nullable = true)
 |-- Education: string (nullable = true)
 |-- Self_Employed: string (nullable = true)
 |-- ApplicantIncome: long (nullable = true)
 |-- CoapplicantIncome: double (nullable = true)
 |-- LoanAmount: double (nullable = true)
 |-- Loan_Amount_Term: double (nullable = true)
 |-- Credit_History: double (nullable = true)
 |-- Property_Area: string (nullable = true)
 |-- Loan_Status: string (nullable = true)


Training Data Sample:
+--------+------+-------+----------+------------+-------------+---------------+-----------------+----------+----------------+--------------+-------------+-----------+
| Loan_ID|Gender|Married|Dependents|   Education|Self_Employed|ApplicantIncome|CoapplicantIncome|LoanAmount|Loan_Amount_Term|Credit_History|